# 02 — Build the IBGE 1 km grid for Petrópolis

This notebook constructs the analytical **1 km × 1 km IBGE grid** for the Petrópolis OpenStreetMap study.

It is based on the empirical diagnostics produced by Notebook 01. Those diagnostics showed that the source file is **mixed-resolution**, not purely 200 m:

- 25,055 total source features;
- 23,864 features matched the expected `200M...` identifier pattern and had areas close to 40,000 m²;
- 1,191 remaining features had areas close to 1,000,000 m²;
- 2,192 unique official `nome_1km` parent identifiers;
- 920 parent groups contained exactly 25 source features and dissolved to approximately 1 km²;
- all input geometries were valid and non-empty;
- both datasets use SIRGAS 2000 / EPSG:4674.

Therefore, this notebook **does not blindly dissolve every source record by `nome_1km`**. It first classifies source resolution, gives priority to an official native 1 km geometry when one is present, otherwise builds the 1 km cell from a complete set of 25 × 200 m cells, and explicitly flags unresolved/incomplete groups.

The source GeoParquet files are never modified.

In [ ]:
from pathlib import Path
import json
import re
import sys
import platform

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 140)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("GeoPandas:", gpd.__version__)
print("Pandas:", pd.__version__)

## 1. Paths and configuration

In [ ]:
def find_project_root(start: Path) -> Path:
    """Find the repository root by locating the Petropolis directory."""
    start = start.resolve()
    for candidate in [start, start.parent, *start.parents]:
        if (candidate / "Petropolis").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root containing 'Petropolis'.")

PROJECT_ROOT = find_project_root(Path.cwd())
INPUT_DIR = PROJECT_ROOT / "Petropolis"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "Petropolis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MUNICIPALITY_FILE = INPUT_DIR / "Petropolis_2025_4674_IBGE.parquet"
GRID_SOURCE_FILE = INPUT_DIR / "petropolis_grade_id26_4674_IBGE.parquet"
OUTPUT_FILE = OUTPUT_DIR / "petropolis_grid_1km.parquet"

SOURCE_CRS_EPSG = 4674
METRIC_CRS = "EPSG:31983"  # SIRGAS 2000 / UTM zone 23S
EXPECTED_200M_AREA_M2 = 40_000.0
EXPECTED_1KM_AREA_M2 = 1_000_000.0
AREA_TOLERANCE = 0.02
FRACTION_TOLERANCE = 1e-9

for path in [MUNICIPALITY_FILE, GRID_SOURCE_FILE]:
    if not path.exists():
        raise FileNotFoundError(f"Required input file not found: {path}")

print("Project root:", PROJECT_ROOT)
print("Output file:", OUTPUT_FILE)

## 2. Load and validate input datasets

In [ ]:
municipality = gpd.read_parquet(MUNICIPALITY_FILE)
grid_source = gpd.read_parquet(GRID_SOURCE_FILE)

print(f"Municipality features: {len(municipality):,}")
print(f"Grid source features: {len(grid_source):,}")
print("Municipality CRS:", municipality.crs)
print("Grid source CRS:", grid_source.crs)

if municipality.crs is None or grid_source.crs is None:
    raise ValueError("Both input datasets must have a defined CRS.")

if municipality.crs != grid_source.crs:
    raise ValueError("Municipality and grid source must use the same CRS.")

if municipality.crs.to_epsg() != SOURCE_CRS_EPSG or grid_source.crs.to_epsg() != SOURCE_CRS_EPSG:
    raise ValueError("Both datasets are expected to use SIRGAS 2000 / EPSG:4674.")

required_columns = {"ID_UNICO", "nome_1km", "geometry"}
missing_columns = required_columns.difference(grid_source.columns)
if missing_columns:
    raise KeyError(f"Missing required grid columns: {sorted(missing_columns)}")

print("PASS — Input datasets and mandatory fields are valid.")

## 3. Classify source resolution

Notebook 01 demonstrated that the source contains both approximately 200 m and approximately 1 km geometries.  
Resolution is classified using **both the official identifier prefix and metric geometry area**.

The identifier remains the primary semantic indicator. Geometry area is used as an independent validation.

In [ ]:
grid_metric = grid_source.to_crs(METRIC_CRS).copy()
grid_metric["source_area_m2"] = grid_metric.geometry.area

id_text = grid_metric["ID_UNICO"].astype("string")

grid_metric["id_resolution"] = np.select(
    [
        id_text.str.match(r"^200M", case=False, na=False),
        id_text.str.match(r"^1KM", case=False, na=False),
    ],
    ["200m", "1km"],
    default="other",
)

grid_metric["area_resolution"] = np.select(
    [
        grid_metric["source_area_m2"].between(
            EXPECTED_200M_AREA_M2 * (1 - AREA_TOLERANCE),
            EXPECTED_200M_AREA_M2 * (1 + AREA_TOLERANCE),
        ),
        grid_metric["source_area_m2"].between(
            EXPECTED_1KM_AREA_M2 * (1 - AREA_TOLERANCE),
            EXPECTED_1KM_AREA_M2 * (1 + AREA_TOLERANCE),
        ),
    ],
    ["200m", "1km"],
    default="other",
)

grid_metric["source_resolution"] = np.where(
    grid_metric["id_resolution"].isin(["200m", "1km"]),
    grid_metric["id_resolution"],
    grid_metric["area_resolution"],
)

resolution_summary = (
    grid_metric.groupby(["source_resolution", "id_resolution", "area_resolution"], dropna=False)
    .size()
    .rename("features")
    .reset_index()
)

display(resolution_summary)

resolution_conflicts = grid_metric[
    grid_metric["id_resolution"].isin(["200m", "1km"])
    & grid_metric["area_resolution"].isin(["200m", "1km"])
    & grid_metric["id_resolution"].ne(grid_metric["area_resolution"])
].copy()

print(f"Resolution conflicts: {len(resolution_conflicts):,}")
if len(resolution_conflicts):
    display(resolution_conflicts[["ID_UNICO", "nome_1km", "source_area_m2", "id_resolution", "area_resolution"]].head(30))

## 4. Separate 200 m source cells and native 1 km cells

In [ ]:
source_200m = grid_metric[grid_metric["source_resolution"].eq("200m")].copy()
source_native_1km = grid_metric[grid_metric["source_resolution"].eq("1km")].copy()
source_other = grid_metric[grid_metric["source_resolution"].eq("other")].copy()

print(f"200 m source cells: {len(source_200m):,}")
print(f"Native 1 km source cells: {len(source_native_1km):,}")
print(f"Unclassified source features: {len(source_other):,}")

if len(source_other):
    display(
        source_other[
            ["ID_UNICO", "nome_1km", "source_area_m2", "id_resolution", "area_resolution"]
        ].head(50)
    )

if source_200m["ID_UNICO"].duplicated().any():
    raise ValueError("Duplicate 200 m source identifiers were detected.")

if source_native_1km["ID_UNICO"].duplicated().any():
    raise ValueError("Duplicate native 1 km source identifiers were detected.")

## 5. Aggregate 200 m cells by the official IBGE `nome_1km`

The official parent identifier is used as the aggregation key.  
A complete 1 km parent derived exclusively from 200 m cells should contain 25 source cells.

In [ ]:
counts_200m = (
    source_200m.groupby("nome_1km")
    .size()
    .rename("n_200m_cells")
    .reset_index()
)

aggregated_200m = (
    source_200m[["nome_1km", "geometry"]]
    .dissolve(by="nome_1km", as_index=False)
    .merge(counts_200m, on="nome_1km", how="left")
)

aggregated_200m["aggregated_area_m2"] = aggregated_200m.geometry.area
aggregated_200m["complete_25"] = aggregated_200m["n_200m_cells"].eq(25)
aggregated_200m["area_near_1km2"] = aggregated_200m["aggregated_area_m2"].between(
    EXPECTED_1KM_AREA_M2 * (1 - AREA_TOLERANCE),
    EXPECTED_1KM_AREA_M2 * (1 + AREA_TOLERANCE),
)
aggregated_200m["valid_aggregated_1km"] = (
    aggregated_200m["complete_25"] & aggregated_200m["area_near_1km2"]
)

display(
    counts_200m["n_200m_cells"]
    .value_counts()
    .sort_index()
    .rename_axis("n_200m_cells")
    .reset_index(name="parent_groups")
)

print(
    "Valid 1 km cells reconstructed from 25 × 200 m cells:",
    f"{aggregated_200m['valid_aggregated_1km'].sum():,}",
)

## 6. Validate native 1 km geometries and parent uniqueness

A native 1 km geometry is considered authoritative for its `nome_1km` parent when:

1. exactly one native 1 km feature exists for that parent;
2. its geometry is valid and non-empty;
3. its metric area is within the configured tolerance around 1 km².

If a native 1 km cell exists for a parent that also contains 200 m records, the native geometry has priority for the final canonical 1 km grid. This avoids creating overlapping duplicate 1 km cells.

In [ ]:
native_counts = (
    source_native_1km.groupby("nome_1km")
    .size()
    .rename("n_native_1km")
    .reset_index()
)

native_1km = (
    source_native_1km.copy()
    .merge(native_counts, on="nome_1km", how="left")
)

native_1km["native_area_near_1km2"] = native_1km["source_area_m2"].between(
    EXPECTED_1KM_AREA_M2 * (1 - AREA_TOLERANCE),
    EXPECTED_1KM_AREA_M2 * (1 + AREA_TOLERANCE),
)
native_1km["valid_native_1km"] = (
    native_1km["n_native_1km"].eq(1)
    & native_1km.geometry.is_valid
    & ~native_1km.geometry.is_empty
    & native_1km["native_area_near_1km2"]
)

display(native_counts["n_native_1km"].value_counts().sort_index().to_frame("parent_groups"))

invalid_native = native_1km[~native_1km["valid_native_1km"]].copy()
print(f"Invalid or ambiguous native 1 km features: {len(invalid_native):,}")
if len(invalid_native):
    display(
        invalid_native[
            ["ID_UNICO", "nome_1km", "source_area_m2", "n_native_1km", "valid_native_1km"]
        ].head(50)
    )

## 7. Build one canonical 1 km geometry per official parent

Priority:

1. **native_1km** — use the official native 1 km geometry when available and valid;
2. **aggregated_200m** — otherwise use the dissolve of exactly 25 valid 200 m cells;
3. **unresolved** — do not invent a full 1 km geometry from an incomplete subset.

Unresolved parents are retained in a diagnostic table. If any unresolved parent intersects Petrópolis, the notebook stops before writing the analytical output, because silently approximating a missing 1 km geometry would compromise spatial reproducibility.

In [ ]:
valid_native = native_1km[native_1km["valid_native_1km"]].copy()
valid_native = valid_native[
    ["nome_1km", "ID_UNICO", "TOTAL", "TOTAL_DOM", "geometry", "source_area_m2"]
].rename(
    columns={
        "ID_UNICO": "native_1km_id",
        "TOTAL": "population_total",
        "TOTAL_DOM": "occupied_households_total",
        "source_area_m2": "cell_area_m2",
    }
)
valid_native["source_method"] = "native_1km"
valid_native["n_200m_cells"] = 0
valid_native["has_native_1km"] = True
valid_native["source_group_complete"] = True

valid_aggregated = aggregated_200m[aggregated_200m["valid_aggregated_1km"]].copy()

# Aggregate census attributes only for parents reconstructed from 200 m cells.
agg_attributes = (
    source_200m.groupby("nome_1km", as_index=False)
    .agg(
        population_total=("TOTAL", "sum"),
        occupied_households_total=("TOTAL_DOM", "sum"),
    )
)

valid_aggregated = valid_aggregated.merge(agg_attributes, on="nome_1km", how="left")
valid_aggregated = valid_aggregated[
    [
        "nome_1km",
        "n_200m_cells",
        "geometry",
        "aggregated_area_m2",
        "population_total",
        "occupied_households_total",
    ]
].rename(columns={"aggregated_area_m2": "cell_area_m2"})
valid_aggregated["native_1km_id"] = pd.NA
valid_aggregated["source_method"] = "aggregated_200m"
valid_aggregated["has_native_1km"] = False
valid_aggregated["source_group_complete"] = True

native_parent_ids = set(valid_native["nome_1km"])
valid_aggregated = valid_aggregated[~valid_aggregated["nome_1km"].isin(native_parent_ids)].copy()

canonical_columns = [
    "nome_1km",
    "native_1km_id",
    "source_method",
    "n_200m_cells",
    "has_native_1km",
    "source_group_complete",
    "population_total",
    "occupied_households_total",
    "cell_area_m2",
    "geometry",
]

canonical_1km_metric = gpd.GeoDataFrame(
    pd.concat(
        [
            valid_native[canonical_columns],
            valid_aggregated[canonical_columns],
        ],
        ignore_index=True,
    ),
    geometry="geometry",
    crs=METRIC_CRS,
)

if canonical_1km_metric["nome_1km"].duplicated().any():
    duplicated = canonical_1km_metric.loc[
        canonical_1km_metric["nome_1km"].duplicated(keep=False), "nome_1km"
    ].unique()
    raise ValueError(f"Duplicate canonical 1 km parent IDs detected: {duplicated[:20]}")

canonical_parent_ids = set(canonical_1km_metric["nome_1km"])

all_parent_ids = set(grid_metric["nome_1km"].dropna())
unresolved_parent_ids = sorted(all_parent_ids.difference(canonical_parent_ids))

unresolved = (
    aggregated_200m[aggregated_200m["nome_1km"].isin(unresolved_parent_ids)]
    .copy()
)

print(f"Canonical 1 km cells: {len(canonical_1km_metric):,}")
print(f"  Native 1 km: {(canonical_1km_metric['source_method'] == 'native_1km').sum():,}")
print(f"  Aggregated from 200 m: {(canonical_1km_metric['source_method'] == 'aggregated_200m').sum():,}")
print(f"Unresolved parent IDs: {len(unresolved_parent_ids):,}")

## 8. Verify unresolved parents against the Petrópolis boundary

In [ ]:
municipality_metric = municipality.to_crs(METRIC_CRS).copy()
municipality_union = municipality_metric.geometry.union_all()

if len(unresolved):
    unresolved["intersects_petropolis"] = unresolved.geometry.intersects(municipality_union)
    unresolved["partial_intersection_area_m2"] = unresolved.geometry.intersection(municipality_union).area
    unresolved_inside = unresolved[unresolved["intersects_petropolis"]].copy()
else:
    unresolved_inside = unresolved.copy()

print(f"Unresolved parent groups intersecting Petrópolis: {len(unresolved_inside):,}")

if len(unresolved_inside):
    display(
        unresolved_inside[
            [
                "nome_1km",
                "n_200m_cells",
                "aggregated_area_m2",
                "partial_intersection_area_m2",
            ]
        ].sort_values(["n_200m_cells", "nome_1km"]).head(100)
    )

    raise RuntimeError(
        "At least one unresolved 1 km parent intersects Petrópolis. "
        "A complete official 1 km geometry cannot be reconstructed safely from the current subset. "
        "Add the official continuous IBGE 1 km grid or expand the source grid before proceeding."
    )

print("PASS — No unresolved parent group affects the Petrópolis analytical extent.")

## 9. Validate the canonical 1 km grid

The geometry remains **unclipped**. This is important because all OSM metrics will later use a constant 1 km² support.

In [ ]:
canonical_1km_metric["cell_area_m2"] = canonical_1km_metric.geometry.area
canonical_1km_metric["area_near_1km2"] = canonical_1km_metric["cell_area_m2"].between(
    EXPECTED_1KM_AREA_M2 * (1 - AREA_TOLERANCE),
    EXPECTED_1KM_AREA_M2 * (1 + AREA_TOLERANCE),
)
canonical_1km_metric["geometry_valid"] = canonical_1km_metric.geometry.is_valid
canonical_1km_metric["geometry_empty"] = canonical_1km_metric.geometry.is_empty

print("Canonical 1 km area statistics (m²):")
display(
    canonical_1km_metric["cell_area_m2"]
    .describe(percentiles=[0.01, 0.05, 0.50, 0.95, 0.99])
    .to_frame("cell_area_m2")
)

print(
    "Cells near 1 km²:",
    f"{canonical_1km_metric['area_near_1km2'].sum():,} / {len(canonical_1km_metric):,}",
)
print(
    "Valid geometries:",
    f"{canonical_1km_metric['geometry_valid'].sum():,} / {len(canonical_1km_metric):,}",
)

## 10. Calculate the spatial relationship with Petrópolis

For each complete 1 km cell, calculate:

- whether the full cell is within the municipality;
- whether it intersects the municipality;
- intersection area in square metres;
- fraction of the full 1 km cell covered by Petrópolis;
- a categorical spatial relation.

The original 1 km cell geometry is preserved.

In [ ]:
canonical_1km_metric["intersects_petropolis"] = canonical_1km_metric.geometry.intersects(
    municipality_union
)
canonical_1km_metric["within_petropolis"] = canonical_1km_metric.geometry.within(
    municipality_union
)
canonical_1km_metric["touches_petropolis"] = canonical_1km_metric.geometry.touches(
    municipality_union
)

intersection_geometry = canonical_1km_metric.geometry.intersection(municipality_union)
canonical_1km_metric["intersection_area_m2"] = intersection_geometry.area
canonical_1km_metric["municipality_fraction"] = (
    canonical_1km_metric["intersection_area_m2"] / canonical_1km_metric["cell_area_m2"]
).clip(lower=0.0, upper=1.0)

canonical_1km_metric["spatial_relation"] = np.select(
    [
        canonical_1km_metric["within_petropolis"]
        | canonical_1km_metric["municipality_fraction"].ge(1.0 - FRACTION_TOLERANCE),
        canonical_1km_metric["intersects_petropolis"]
        & canonical_1km_metric["intersection_area_m2"].gt(0),
        canonical_1km_metric["touches_petropolis"],
    ],
    ["within", "boundary_intersection", "touches_only"],
    default="outside",
)

display(
    canonical_1km_metric["spatial_relation"]
    .value_counts()
    .rename_axis("spatial_relation")
    .reset_index(name="n_1km_cells")
)

## 11. Select the 1 km cells for the Petrópolis study

The analytical selection includes all complete canonical 1 km cells that **intersect** the Petrópolis municipal geometry.  
The full square geometry is retained; cells are not clipped at the municipal boundary.

This design keeps a constant spatial support for later ohsome API metrics while `municipality_fraction` records how much of each cell lies inside Petrópolis.

In [ ]:
petropolis_grid_1km_metric = canonical_1km_metric[
    canonical_1km_metric["intersects_petropolis"]
].copy()

petropolis_grid_1km_metric = petropolis_grid_1km_metric.rename(
    columns={"nome_1km": "grid_1km_id"}
)

petropolis_grid_1km_metric = petropolis_grid_1km_metric.sort_values(
    "grid_1km_id"
).reset_index(drop=True)

print(f"Selected 1 km cells intersecting Petrópolis: {len(petropolis_grid_1km_metric):,}")
display(
    petropolis_grid_1km_metric[
        [
            "grid_1km_id",
            "source_method",
            "n_200m_cells",
            "cell_area_m2",
            "intersection_area_m2",
            "municipality_fraction",
            "spatial_relation",
        ]
    ].head(20)
)

## 12. Quality-control summary

In [ ]:
def qc_status(condition: bool, warning: bool = False) -> str:
    if condition:
        return "PASS"
    return "WARNING" if warning else "FAIL"

qc_rows = [
    [
        "Input CRS is EPSG:4674",
        qc_status(grid_source.crs.to_epsg() == 4674 and municipality.crs.to_epsg() == 4674),
        str(grid_source.crs.to_epsg()),
    ],
    [
        "No resolution conflicts",
        qc_status(len(resolution_conflicts) == 0, warning=True),
        f"{len(resolution_conflicts):,} conflicts",
    ],
    [
        "No unclassified source features",
        qc_status(len(source_other) == 0, warning=True),
        f"{len(source_other):,} unclassified",
    ],
    [
        "Canonical 1 km IDs are unique",
        qc_status(petropolis_grid_1km_metric["grid_1km_id"].is_unique),
        f"{petropolis_grid_1km_metric['grid_1km_id'].nunique():,} unique IDs",
    ],
    [
        "All selected geometries are valid",
        qc_status(petropolis_grid_1km_metric.geometry.is_valid.all()),
        f"{(~petropolis_grid_1km_metric.geometry.is_valid).sum():,} invalid",
    ],
    [
        "All selected geometries are non-empty",
        qc_status((~petropolis_grid_1km_metric.geometry.is_empty).all()),
        f"{petropolis_grid_1km_metric.geometry.is_empty.sum():,} empty",
    ],
    [
        "All selected cells are approximately 1 km²",
        qc_status(petropolis_grid_1km_metric["area_near_1km2"].all()),
        f"{petropolis_grid_1km_metric['area_near_1km2'].mean():.2%} within ±{AREA_TOLERANCE:.0%}",
    ],
    [
        "All selected cells intersect Petrópolis",
        qc_status(petropolis_grid_1km_metric["intersects_petropolis"].all()),
        f"{petropolis_grid_1km_metric['intersects_petropolis'].sum():,} / {len(petropolis_grid_1km_metric):,}",
    ],
    [
        "Municipality fractions are bounded [0, 1]",
        qc_status(petropolis_grid_1km_metric["municipality_fraction"].between(0, 1).all()),
        f"min={petropolis_grid_1km_metric['municipality_fraction'].min():.6f}, max={petropolis_grid_1km_metric['municipality_fraction'].max():.6f}",
    ],
    [
        "No unresolved parent intersects Petrópolis",
        qc_status(len(unresolved_inside) == 0),
        f"{len(unresolved_inside):,} unresolved intersecting parents",
    ],
]

qc_table = pd.DataFrame(qc_rows, columns=["check", "status", "value"])
display(qc_table)
display(
    qc_table["status"]
    .value_counts()
    .reindex(["PASS", "WARNING", "FAIL"], fill_value=0)
    .to_frame("count")
)

if (qc_table["status"] == "FAIL").any():
    raise RuntimeError("Quality control failed. The output GeoParquet will not be written.")

## 13. Diagnostic map

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

petropolis_grid_1km_metric.plot(
    ax=ax,
    column="spatial_relation",
    categorical=True,
    legend=True,
    alpha=0.55,
    linewidth=0.5,
    edgecolor="black",
)
municipality_metric.boundary.plot(ax=ax, linewidth=1.5)

ax.set_title("Canonical 1 km IBGE grid intersecting Petrópolis")
ax.set_axis_off()
plt.show()

## 14. Save the processed GeoParquet

The output is saved in **SIRGAS 2000 / EPSG:4674**, matching the official IBGE source data.

A later notebook dedicated to the ohsome API should create an EPSG:4326 copy only for API requests. The analytical master grid remains in EPSG:4674.

The output geometry is the complete 1 km cell, not the municipality-clipped intersection.

In [ ]:
output_columns = [
    "grid_1km_id",
    "native_1km_id",
    "source_method",
    "n_200m_cells",
    "has_native_1km",
    "source_group_complete",
    "population_total",
    "occupied_households_total",
    "cell_area_m2",
    "intersection_area_m2",
    "municipality_fraction",
    "spatial_relation",
    "intersects_petropolis",
    "within_petropolis",
    "touches_petropolis",
    "geometry",
]

petropolis_grid_1km = (
    petropolis_grid_1km_metric[output_columns]
    .to_crs(f"EPSG:{SOURCE_CRS_EPSG}")
    .copy()
)

petropolis_grid_1km.to_parquet(
    OUTPUT_FILE,
    index=False,
    compression="zstd",
)

print(f"Saved: {OUTPUT_FILE}")
print(f"Features: {len(petropolis_grid_1km):,}")
print("CRS:", petropolis_grid_1km.crs)
print("File size:", f"{OUTPUT_FILE.stat().st_size / 1024:.1f} KiB")

## 15. Re-open and verify the written GeoParquet

This final read-back check confirms that the file can be loaded and that its critical metadata survived serialization.

In [ ]:
written = gpd.read_parquet(OUTPUT_FILE)

verification = pd.Series(
    {
        "features_match": len(written) == len(petropolis_grid_1km),
        "crs_epsg_4674": written.crs is not None and written.crs.to_epsg() == 4674,
        "grid_id_unique": written["grid_1km_id"].is_unique,
        "no_null_geometry": written.geometry.notna().all(),
        "no_empty_geometry": (~written.geometry.is_empty).all(),
        "geometry_valid": written.geometry.is_valid.all(),
        "all_intersect_petropolis": written["intersects_petropolis"].all(),
    },
    name="PASS",
)

display(verification)

if not verification.all():
    raise RuntimeError("Written GeoParquet failed the final read-back verification.")

print("PASS — Processed 1 km GeoParquet is ready for the next stage.")

## 16. Output contract for the ohsome stage

The next stage should treat `grid_1km_id` as the permanent spatial key.

For ohsome API requests:

1. load `data/processed/Petropolis/petropolis_grid_1km.parquet`;
2. reproject a request copy to EPSG:4326;
3. preserve `grid_1km_id` in every request/response table;
4. do not clip the 1 km cells;
5. keep `municipality_fraction` for sensitivity analyses involving boundary cells;
6. store raw API responses separately from processed analytical tables.

Recommended first OSM filter: `highway=*`.